# Baseline Kimi Linear Model


In [ ]:
!pip uninstall -y jax jaxlib jax-cuda12-plugin jax-cuda13-plugin jax-cuda12-pjrt jax-cuda13-pjrt 2>/dev/null
!pip install -U "jax[cuda13]"

In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import functools
import pickle
import sys
from typing import TYPE_CHECKING, Any, Callable, Iterator, Literal, Optional, NamedTuple, Tuple
from dataclasses import dataclass, asdict
import flax.linen as nn
import optax
import wandb
from pathlib import Path
import optax
from optax import GradientTransformation


if TYPE_CHECKING:
    from transformers import PreTrainedTokenizerBase
else:
    PreTrainedTokenizerBase = Any


if TYPE_CHECKING:
    from datasets import IterableDataset
else:
    IterableDataset = Any
from einops import rearrange

print(jax.__version__)
print(jax.default_backend())


### Configurations





  - Standard Kimi Linear stack
  - Pattern per layer group: Bottom, Bottom, Bottom, Top
  - BottomBlock = KDA + MoE
  - TopBlock = MLA + MoE
  - 9 BottomBlocks: KDA + MoE
  - 3 TopBlocks: MLA + MoE





In [12]:

@dataclass
class ModelArg:
    """
    Shared config for all three model variants and the trainer.
    """


    dim: int = 1024
    inter_dim: int = 1536
    heads: int = 8
    local_heads: int = 8
    head_dim: int = 128
    vocab_size: int = 32768

    # MLA
    q_lora_rank: int = 256
    kv_lora_rank: int = 64
    qk_head_dim: int = 128
    qk_nope_head_dim: int = 64
    qk_rope_head_dim: int = 64
    v_head_dim: int = 128

    # MoE
    n_routed_experts: int = 4
    n_activated_experts: int = 2
    n_groups: int = 1
    topk_groups: int = 1
    shared_experts: int = 1
    score_func: Literal["softmax", "sigmoid"] = "sigmoid"
    scaling_factor: float = 2.446

    # Architecture
    n_layer_groups: int = 1
    block_size: int = 128
    max_seq: int = 1024

    # Data
    dataset_name: str = "HuggingFaceFW/fineweb-edu"
    dataset_subset: str = "sample-10BT"
    dataset_split: str = "train"
    streaming: bool = True
    text_field: str = "text"
    seed: int = 0
    shuffle_buffer_size: int = 10_000
    max_examples: int | None = None

    # Training
    model_type: Literal["baseline", "full", "batch"] = "baseline"
    tokenizer_name: str = "gpt2"
    use_wandb: bool = True
    wandb_project: str = "attn-residual"
    wandb_entity: str | None = None
    wandb_run_name: str | None = None
    log_every_steps: int = 10
    residual_log_every_steps: int = 50

    lr: float = 3e-4
    min_lr: float = 1e-5
    batch_size: int = 4


    total_steps: int = 10_000


    warmup_steps: int = 300
    hold_steps: int = 700

    # Muon
    muon_beta: float = 0.95
    muon_ns_steps: int = 5
    muon_lr: float = 0.01

    # Adam
    adam_b1: float = 0.9
    adam_b2: float = 0.95
    adam_eps: float = 1e-8
    weight_decay: float = 0.01

    # Grad clipping
    max_grad_norm: float = 1.0

# Utils

def exists(x):
  return x is not None

def default(val, d):
  if exists(val):
    return val
  return d() if callable(d) else d

# RMSNorm

class RMSNorm(nn.Module):
  """
  Root Mean Squared Normalization
  """
  dim: int
  eps: float = 1e-6

  def setup(self):
      self.weight = self.param('weight', nn.initializers.ones, (self.dim,))

  def __call__(self, x: jnp.ndarray) -> jnp.ndarray:

      orig_dtype = x.dtype
      x = x.astype(jnp.float32)
      rms = jnp.sqrt(jnp.mean(x ** 2, axis=-1, keepdims=True) + self.eps)
      return ((x / rms) * self.weight).astype(orig_dtype)

# Linear(s)

def linear(x: jnp.ndarray, weights: jnp.ndarray, bias: Optional[jnp.ndarray] = None) -> jnp.ndarray:
  out = jnp.dot(x, weights)
  if exists(bias):
    out = out + bias
  return out


class Linear(nn.Module):
  in_features: int
  out_features: int
  use_bias: bool = False

  @nn.compact
  def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
    weights = self.param(
        'weights',
        nn.initializers.lecun_normal(),
        (self.in_features, self.out_features)
    )
    bias = None
    if self.use_bias:
      bias = self.param(
          'bias',
          nn.initializers.zeros,
          (self.out_features,)
      )
    return linear(x, weights, bias)



# Short Convolution

class ShortConv(nn.Module):
    features: int
    kernel_size: int = 4

    def setup(self):
        self.w = self.param('w', nn.initializers.lecun_normal(),
                            (self.features, self.kernel_size))
        self.b = self.param('b', nn.initializers.zeros,
                            (self.features,))

    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        batch, seq, dim = x.shape
        k = self.kernel_size
        x_padded = jnp.pad(x, ((0, 0), (k - 1, 0), (0, 0)))
        out = jnp.zeros((batch, seq, dim))
        for i in range(k):
            out = out + x_padded[:, i:i+seq, :] * self.w[None, None, :, i]
        return out + self.b



# Chunked Kimi Delta Attention

class KDA(nn.Module):
  """
  Chunked Kimi Delta Attention
  """
  dim: int
  heads: int
  head_dim: int = 128
  chunk_size: int = 64

  @nn.compact
  def __call__(self,
               q: jnp.ndarray,
               k: jnp.ndarray,
               v: jnp.ndarray,
               g: jnp.ndarray,
               beta: jnp.ndarray,
               initial_state: Optional[jnp.ndarray] = None,
               ) -> tuple[jnp.ndarray, jnp.ndarray]:

      B, T, H, K = q.shape
      V = v.shape[-1]
      C = self.chunk_size
      N = T // C
      assert T % C == 0


      q, k, v, g, beta = [x.astype(jnp.bfloat16) for x in (q, k, v, g, beta)]



      q, k, v, g = [rearrange(x, 'b (n c) h d -> b h n c d', c=C)
                    for x in (q, k, v, g)]
      beta = rearrange(beta, 'b (n c) h -> b h n c', c=C)

      q = q * (K ** -0.5)
      g = jnp.cumsum(g, axis=-2)
      g = jnp.clip(g, -8.0, 8.0)


      A = jnp.einsum('...jd,...id->...ji',
                      k * jnp.exp(g),
                      k * jnp.exp(-g),
                      precision=jax.lax.Precision.HIGHEST)

      A = A * beta[..., None]


      mask_diag = jnp.tril(jnp.ones((C, C), dtype=jnp.bool_))
      A = -jnp.where(mask_diag, A, 0.0)

      def fwd_sub_step(i, A):
        row = jax.lax.dynamic_slice_in_dim(A, i, 1, axis=-2)
        update = jnp.matmul(row, A)
        row = row + update
        return jax.lax.dynamic_update_slice_in_dim(A, row, i, axis=-2)

      A = jax.lax.fori_loop(1, C, fwd_sub_step, A)


      A = (A + jnp.eye(C, dtype=jnp.bfloat16)) * beta[..., None, :]


      w = jnp.einsum('...ij,...jd->...id', A, jnp.exp(g) * k,
                      precision=jax.lax.Precision.HIGHEST)
      u = jnp.einsum('...ij,...jd->...id', A, v,
                      precision=jax.lax.Precision.HIGHEST)

      S_init = jnp.zeros((B, H, K, V), dtype=jnp.bfloat16)
      if initial_state is not None:
          S_init = S_init + initial_state.astype(jnp.bfloat16)

      mask_strict = jnp.triu(jnp.ones((C, C), dtype=jnp.bool_), k=1)

      def chunk_step(S, inputs):
          q_i, k_i, u_i, g_i, w_i = inputs


          A_qk = jnp.einsum('...id,...jd->...ij',
                             q_i * jnp.exp(g_i),
                             k_i * jnp.exp(-g_i),
                             precision=jax.lax.Precision.HIGHEST)
          A_qk = jnp.where(mask_strict, 0.0, A_qk)


          v_i = u_i - jnp.einsum('...id,...dv->...iv', w_i, S,
                                  precision=jax.lax.Precision.HIGHEST)


          o_i = (jnp.einsum('...id,...dv->...iv',
                             q_i * jnp.exp(g_i), S,
                             precision=jax.lax.Precision.HIGHEST)
               + jnp.einsum('...ij,...jv->...iv',
                             A_qk, v_i,
                             precision=jax.lax.Precision.HIGHEST))


          decay = g_i[..., -1, :]
          S_new = (S * jnp.exp(jnp.clip(decay[..., None], -8.0, 0.0))
                    + jnp.einsum('...id,...iv->...dv',
                    k_i * jnp.exp(-g_i), v_i,
                    precision=jax.lax.Precision.HIGHEST))



          return S_new, o_i


      scan_inputs = tuple(jnp.moveaxis(x, 2, 0) for x in (q, k, u, g, w))

      S_final, o = jax.lax.scan(
          chunk_step,
          S_init,
          scan_inputs
      )

      return (
          rearrange(o, 'n b h c v -> b (n c) h v').astype(jnp.bfloat16),
          S_final
      )


# Multi Linear Attention

class multihead_attn(nn.Module):
  """
  Multi-head Latent Attention (naive variant, no KV cache)

  Uses LoRA-compressed Q projection and joint KV compression.
  Q = [q_nope, q_pe], K = [k_nope, k_pe], split into rope/non-rope parts.
  """
  dim: int
  heads: int
  local_heads: int
  q_lora_rank: int
  kv_lora_rank: int
  qk_head_dim: int = 128
  v_head_dim: int = 128
  qk_nope_head_dim: int = 64
  qk_rope_head_dim: int = 64
  max_seq: int = 4096

  def setup(self):
    self.softmax_scale = self.qk_head_dim ** -0.5

    if self.q_lora_rank == 0:
      self.wq = Linear(in_features=self.dim, out_features=self.heads * self.qk_head_dim)
    else:
      self.wq_a = Linear(in_features=self.dim, out_features=self.q_lora_rank)
      self.q_norm = RMSNorm(self.q_lora_rank)
      self.wq_b = Linear(in_features=self.q_lora_rank, out_features=self.heads * self.qk_head_dim)

    self.wkv_a = Linear(in_features=self.dim, out_features=self.kv_lora_rank + self.qk_rope_head_dim)
    self.kv_norm = RMSNorm(self.kv_lora_rank)
    self.wkv_b = Linear(in_features=self.kv_lora_rank,
                        out_features=self.heads * (self.qk_nope_head_dim + self.v_head_dim))
    self.wo = Linear(in_features=self.heads * self.v_head_dim, out_features=self.dim)

  def __call__(self, x: jnp.ndarray, start: int = 0,
               freqs_cis: Optional[jnp.ndarray] = None,
               mask: Optional[jnp.ndarray] = None):
    batch, seqlen, _ = x.shape

    # Q projection (direct or LoRA-compressed)
    if self.q_lora_rank == 0:
      q = self.wq(x)
    else:
      q = self.wq_b(self.q_norm(self.wq_a(x)))

    q = q.reshape(batch, seqlen, self.local_heads, self.qk_head_dim)
    q_nope, q_pe = jnp.split(q, [self.qk_nope_head_dim], axis=-1)


    kv = self.wkv_a(x)
    kv, k_pe = jnp.split(kv, [self.kv_lora_rank], axis=-1)


    q = jnp.concatenate([q_nope, q_pe], axis=-1)

    kv = self.wkv_b(self.kv_norm(kv))
    kv = kv.reshape(batch, seqlen, self.local_heads, self.qk_nope_head_dim + self.v_head_dim)

    k_nope, v = jnp.split(kv, [self.qk_nope_head_dim], axis=-1)
    k = jnp.concatenate([k_nope, jnp.broadcast_to(
        k_pe[:, :, None, :],
        (batch, seqlen, self.local_heads, self.qk_rope_head_dim))], axis=-1)

    attn_bias = None
    attn_mask = None
    if mask is not None:
      if mask.dtype == jnp.bool_:
        attn_mask = mask
      else:

        if mask.ndim == 2:
          attn_bias = mask[None, None, :, :]
        else:
          attn_bias = mask

    attn_impl = "cudnn" if jax.default_backend() == "gpu" else "xla"
    out = jax.nn.dot_product_attention(
        q.astype(jnp.float16),
        k.astype(jnp.float16),
        v.astype(jnp.float16),
        bias=attn_bias,
        mask=attn_mask,
        scale=self.softmax_scale,
        is_causal=True,
        implementation=attn_impl,
    )
    return self.wo(out.reshape(batch, seqlen, -1))







# Gate - Routes to the right MoE

class Gate(nn.Module):
  """
  Gate - Mechanism that routes to the right MoE Expert

  Routes tokens to top-k experts via 2-stage hierarchical selection:
    1. Score all experts via sigmoid/softmax
    2. Select top-k groups by top-2 expert score sum per group
    3. Select top-k experts from surviving groups
  Bias is per-expert for auxiliary-loss-free load balancing (DeepSeek V3).
  """
  dim: int
  n_routed_experts: int
  topk: int
  n_groups: int
  topk_groups: int
  route_scale: float
  score_func: str = 'sigmoid'



  @nn.compact
  def __call__(self, x: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:

    weight = self.param(
        'weight',
        nn.initializers.normal(stddev=0.01),
        (self.n_routed_experts, self.dim)
    )

    e_score_correction_bias = self.param(
        'e_score_correction_bias',
        nn.initializers.zeros,
        (self.n_routed_experts,)
    )

    scores = linear(x, weight.T)

    if self.score_func == "softmax":
      scores = jax.nn.softmax(scores, axis=-1).astype(jnp.float32)
    else:
      scores = jax.nn.sigmoid(scores)

    original_scores = scores

    # Add per-expert bias for routing decisions only
    scores = scores + e_score_correction_bias

    if self.n_groups > 1:
      scores = scores.reshape(x.shape[0], self.n_groups, -1)

      # Group score: sum of top-2 expert scores within each group
      group_score = jnp.sort(scores, axis=-1)[..., -2:].sum(axis=-1)

      group_idx = jnp.argsort(group_score, axis=-1)[..., -self.topk_groups:]

      mask = jnp.ones((x.shape[0], self.n_groups), dtype=bool)
      mask = mask.at[jnp.arange(x.shape[0])[:, None], group_idx].set(False)

      scores = jnp.where(mask[..., None], -jnp.inf, scores).reshape(x.shape[0], -1)

    # Select top-k experts
    indices = jnp.argsort(scores, axis=-1)[..., -self.topk:]

    # Final weights from original (un-biased) scores
    weights = original_scores[jnp.arange(x.shape[0])[:, None], indices]

    if self.score_func == "sigmoid":
      weights = weights / (weights.sum(axis=-1, keepdims=True) + 1e-20)

    weights = weights * self.route_scale

    return weights, indices


# Expert - The Expert in MoE

class Expert(nn.Module):
    dim: int
    inter_dim: int

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        gate = Linear(self.dim, self.inter_dim)(x)
        up   = Linear(self.dim, self.inter_dim)(x)
        down = Linear(self.inter_dim, self.dim)
        return down(nn.silu(gate) * up)

# MLP - Multi-Layer Perceptron
class MLP(nn.Module):
    dim: int
    inter_dim: int

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        gate = Linear(self.dim, self.inter_dim)(x)
        up   = Linear(self.dim, self.inter_dim)(x)
        down = Linear(self.inter_dim, self.dim)
        return down(nn.silu(gate) * up)


# MoE - Mixture of Experts Layer

class MixtureOfExperts(nn.Module):

    dim: int
    n_routed_experts: int
    n_groups: int
    n_activated_experts: int
    n_shared_experts: int
    topk_groups: int
    inter_dim: int
    route_scale: float
    score_func: str


    def setup(self):
        self.gate = Gate(dim=self.dim, n_routed_experts=self.n_routed_experts,
                         topk=self.n_activated_experts,
                         n_groups=self.n_groups, topk_groups=self.topk_groups,
                         route_scale=self.route_scale, score_func=self.score_func
                         )
        # Vmapped expert: params stacked along axis 0, single call for all experts
        self.experts = nn.vmap(
            Expert,
            variable_axes={'params': 0},
            split_rngs={'params': True},
            in_axes=None, out_axes=0,
            axis_size=self.n_routed_experts
        )(self.dim, self.inter_dim)

        self.shared_experts = MLP(self.dim, self.n_shared_experts * self.inter_dim)

    def __call__(self, x: jnp.ndarray):
        shape = x.shape
        x = x.reshape(-1, self.dim)

        weights, indices = self.gate(x)

        # All experts in parallel: (n_experts, n_tokens, dim)
        all_expert_out = self.experts(x)
        all_expert_out = jnp.swapaxes(all_expert_out, 0, 1)

        # Gather selected expert outputs: (n_tokens, topk, dim)
        selected_out = jnp.take_along_axis(
            all_expert_out,
            indices[..., None],
            axis=1,
        )

        # Weighted sum over topk: (n_tokens, dim)
        y = (selected_out * weights[..., None]).sum(axis=1)

        z = self.shared_experts(x)
        return (y + z).reshape(shape)


# BottomBlock: KDA + MoE (3 of these per TopBlock)

class BottomBlock(nn.Module):
  """KDA_Layer followed by MoE_Layer"""
  dim: int
  heads: int
  head_dim: int
  kernel_size: int
  n_routed_experts: int
  n_groups: int
  n_activated_experts: int
  n_shared_experts: int
  topk_groups: int
  inter_dim: int
  route_scale: float
  score_func: str

  def setup(self):
    self.kda = KDA_Layer(dim=self.dim, heads=self.heads,
                         head_dim=self.head_dim, kernel_size=self.kernel_size)
    self.moe = MoE_Layer(dim=self.dim, n_routed_experts=self.n_routed_experts,
                         n_groups=self.n_groups, n_activated_experts=self.n_activated_experts,
                         n_shared_experts=self.n_shared_experts, topk_groups=self.topk_groups,
                         inter_dim=self.inter_dim, route_scale=self.route_scale,
                         score_func=self.score_func)


  def __call__(self, x: jnp.ndarray, kda_state: Optional[jnp.ndarray] = None):
    x, S = self.kda(x, kda_state)
    x = self.moe(x)
    return x, S


# TopBlock: MLA + MoE (1 of these per 3 BottomBlocks)

class TopBlock(nn.Module):
  """MLA_Layer followed by MoE_Layer"""
  dim: int
  heads: int
  local_heads: int
  q_lora_rank: int
  kv_lora_rank: int
  n_routed_experts: int
  n_groups: int
  n_activated_experts: int
  n_shared_experts: int
  topk_groups: int
  inter_dim: int
  route_scale: float
  score_func: str
  qk_head_dim: int = 128
  v_head_dim: int = 128

  def setup(self):
    self.mla = MLA_Layer(dim=self.dim, heads=self.heads,
                         local_heads=self.local_heads,
                         q_lora_rank=self.q_lora_rank,
                         kv_lora_rank=self.kv_lora_rank,
                         qk_head_dim=self.qk_head_dim,
                         v_head_dim=self.v_head_dim)
    self.moe = MoE_Layer(dim=self.dim, n_routed_experts=self.n_routed_experts,
                         n_groups=self.n_groups, n_activated_experts=self.n_activated_experts,
                         n_shared_experts=self.n_shared_experts, topk_groups=self.topk_groups,
                         inter_dim=self.inter_dim, route_scale=self.route_scale,
                         score_func=self.score_func)

  def __call__(self, x: jnp.ndarray, start: int = 0, freqs_cis: Optional[jnp.ndarray] = None, mask: Optional[jnp.ndarray] = None):
    x = self.mla(x, start, freqs_cis, mask)
    x = self.moe(x)
    return x



# KDA Layer

class KDA_Layer(nn.Module):
    dim: int
    heads: int
    head_dim: int = 128
    kernel_size: int = 4

    def setup(self):
        self.norm_in  = RMSNorm(dim=self.dim)
        self.norm_out = RMSNorm(dim=self.head_dim)
        self.wq       = nn.Dense(self.heads * self.head_dim, use_bias=False)
        self.wk       = nn.Dense(self.heads * self.head_dim, use_bias=False)
        self.wv       = nn.Dense(self.heads * self.head_dim, use_bias=False)
        self.w_beta   = nn.Dense(self.heads, use_bias=False)
        self.alpha_down = nn.Dense(self.head_dim, use_bias=False)
        self.alpha_up   = nn.Dense(self.heads * self.head_dim, use_bias=False)
        self.gate_down  = nn.Dense(self.head_dim, use_bias=False)
        self.gate_up    = nn.Dense(self.heads * self.head_dim, use_bias=False)
        self.wo         = nn.Dense(self.dim, use_bias=False)
        self.kda_attn   = KDA(dim=self.dim, heads=self.heads, head_dim=self.head_dim)

        self.q_conv = ShortConv(features=self.heads * self.head_dim, kernel_size=self.kernel_size)
        self.k_conv = ShortConv(features=self.heads * self.head_dim, kernel_size=self.kernel_size)

    def __call__(self, x, initial_state=None):
        batch, seq, _ = x.shape

        x_norm = self.norm_in(x)
        q = nn.swish(self.q_conv(self.wq(x_norm)))
        q = q / (jnp.linalg.norm(q, axis=-1, keepdims=True) + 1e-6)
        k = nn.swish(self.k_conv(self.wk(x_norm)))
        k = k / (jnp.linalg.norm(k, axis=-1, keepdims=True) + 1e-6)
        v = nn.swish(self.wv(x_norm))

        q = q.reshape(batch, seq, self.heads, self.head_dim)
        k = k.reshape(batch, seq, self.heads, self.head_dim)
        v = v.reshape(batch, seq, self.heads, self.head_dim)

        beta  = nn.sigmoid(self.w_beta(x_norm))
        alpha = nn.log_sigmoid(
            self.alpha_up(self.alpha_down(x_norm))
        ).reshape(batch, seq, self.heads, self.head_dim)
        gate  = nn.sigmoid(
            self.gate_up(self.gate_down(x_norm))
        ).reshape(batch, seq, self.heads, self.head_dim)

        kda_out, S = self.kda_attn(q, k, v, alpha, beta, initial_state)

        out = self.norm_out(kda_out)
        out = (out * gate).reshape(batch, seq, self.heads * self.head_dim)
        return x + self.wo(out), S


# MoE Layer
class MoE_Layer(nn.Module):
  """
  Mixture of Experts Layer = RMSNorm + MoE + residual
  """
  dim: int
  n_routed_experts: int
  n_groups: int
  n_activated_experts: int
  n_shared_experts: int
  topk_groups: int
  inter_dim: int
  route_scale: float
  score_func: str

  def setup(self):
    self.norm = RMSNorm(self.dim)
    self.moe = MixtureOfExperts(dim=self.dim,
                                n_routed_experts=self.n_routed_experts,
                                n_groups=self.n_groups,
                                n_activated_experts=self.n_activated_experts,
                                n_shared_experts=self.n_shared_experts,
                                topk_groups=self.topk_groups,
                                inter_dim=self.inter_dim,
                                route_scale=self.route_scale,
                                score_func=self.score_func
                                )

  def __call__(self, x: jnp.ndarray):
    return x + self.moe(self.norm(x))


# MLA Layer

class MLA_Layer(nn.Module):
  """
  Multi-head Latent Attention Layer = RMSNorm + MLA + residual
  """
  dim: int
  heads: int
  local_heads: int
  q_lora_rank: int
  kv_lora_rank: int
  qk_head_dim: int = 128
  v_head_dim: int = 128

  def setup(self):
    self.norm = RMSNorm(self.dim)
    self.mla = multihead_attn(dim=self.dim,
                              heads=self.heads,
                              local_heads=self.local_heads,
                              q_lora_rank=self.q_lora_rank,
                              kv_lora_rank=self.kv_lora_rank,
                              qk_head_dim=self.qk_head_dim,
                              v_head_dim=self.v_head_dim)

  def __call__(self, x: jnp.ndarray, start: int = 0, freqs_cis: Optional[jnp.ndarray] = None, mask: Optional[jnp.ndarray] = None):
    return x + self.mla(self.norm(x), start, freqs_cis, mask)


# Kimi Linear Transformer: 3x BottomBlock (KDA+MoE) per 1x TopBlock (MLA+MoE)

class Transformer(nn.Module):
  """
  Kimi Linear Transformer

  Interleaves KDA and MLA layers in a 3:1 ratio, each followed by MoE.
  Pattern per group: [Bottom, Bottom, Bottom, Top] x n_groups
  """
  args: ModelArg

class Transformer(nn.Module):
    args: ModelArg

    def setup(self):
        moe_kw = dict(
            n_routed_experts=self.args.n_routed_experts,
            n_groups=self.args.n_groups,
            n_activated_experts=self.args.n_activated_experts,
            n_shared_experts=self.args.shared_experts,
            topk_groups=self.args.topk_groups,
            inter_dim=self.args.inter_dim,
            route_scale=self.args.scaling_factor,
            score_func=self.args.score_func,
        )

        bottom_layers = []
        top_layers = []
        for g in range(self.args.n_layer_groups):
            for _ in range(3):
                bottom_layers.append(BottomBlock(
                    dim=self.args.dim, heads=self.args.heads,
                    head_dim=self.args.head_dim, kernel_size=4,
                    **moe_kw))
            top_layers.append(TopBlock(
                dim=self.args.dim, heads=self.args.heads,
                local_heads=self.args.local_heads,
                q_lora_rank=self.args.q_lora_rank,
                kv_lora_rank=self.args.kv_lora_rank,
                qk_head_dim=self.args.qk_head_dim,
                v_head_dim=self.args.v_head_dim,
                **moe_kw))

        self.bottom_layers = bottom_layers
        self.top_layers = top_layers
        self.norm = RMSNorm(self.args.dim)
        self.tok_emb = nn.Embed(self.args.vocab_size, self.args.dim)
        self.head = nn.Dense(self.args.vocab_size, use_bias=False, name='lm_head')

    def __call__(self, tokens, start=0, freqs_cis=None, mask=None):
        x = self.tok_emb(tokens).astype(jnp.bfloat16)


        b_idx, t_idx = 0, 0
        for g in range(self.args.n_layer_groups):
            for _ in range(3):
                x, _ = self.bottom_layers[b_idx](x)
                b_idx += 1
            x = self.top_layers[t_idx](x, start, freqs_cis, mask)
            t_idx += 1

        x = self.norm(x)
        return self.head(x.astype(jnp.float32))

In [13]:

# Muon Optimizer


def newton_schulz(g: jnp.ndarray, steps: int = 5, eps: float = 1e-7):
  """
  Newton-Schultz iteration on 2D matrix
  """
  assert g.ndim >= 2

  a, b, c = 3.4445, -4.7750, 2.0315

  x = g.astype(jnp.float32)

  x = x / (jnp.linalg.norm(x, axis=(-2, -1), keepdims=True) + eps)

  def _ns_step(_, x):
      A = x @ x.T
      B = b * A + c * A @ A
      return a * x + B @ x

  x = jax.lax.fori_loop(0, steps, _ns_step, x)

  if g.shape[-2] > g.shape[-1]:
      x = x.T

  return x

class MuonState(NamedTuple):
    momentum_buffer: Any
    count: jnp.ndarray


def init_muon(params: Any) -> MuonState:
    return MuonState(
        momentum_buffer=jax.tree_util.tree_map(
            lambda p: jnp.zeros_like(p, dtype=jnp.bfloat16), params
        ),
        count=jnp.zeros([], jnp.int32),
    )


def muon_step(
    grads: Any,
    state: MuonState,
    learning_rate: float = 0.02,
    beta: float = 0.95,
    nesterov: bool = True,
    ns_steps: int = 5,
) -> tuple[Any, MuonState]:
    results = jax.tree_util.tree_map(
        lambda g, b: muon_update(g, b, beta=beta, ns_steps=ns_steps, nesterov=nesterov),
        grads, state.momentum_buffer,
    )
    updates = jax.tree_util.tree_map(lambda r: learning_rate * r[0], results)
    new_buf = jax.tree_util.tree_map(lambda r: r[1], results)

    return updates, MuonState(momentum_buffer=new_buf, count=state.count + 1)



def muon_update(
    grad: jnp.ndarray,
    buf: jnp.ndarray,
    beta: float = 0.95,
    ns_steps: int = 5,
    nesterov: bool = True,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    g = grad.astype(jnp.float32)
    b = buf.astype(jnp.float32)

    new_buf = beta * b + (1.0 - beta) * g
    update = beta * new_buf + (1.0 - beta) * g if nesterov else new_buf

    if grad.ndim == 2:
        transposed = update.shape[0] > update.shape[1]
        if transposed:
            update = update.T
        update = newton_schulz(update, steps=ns_steps)
        if transposed:
            update = update.T
        m, n = grad.shape
        update = update * max(1, m / n) ** 0.5

    return update.astype(grad.dtype), new_buf.astype(buf.dtype)


def _muon_transform(beta: float = 0.95, ns_steps: int = 5) -> GradientTransformation:

    def init_fn(params):
        flat, treedef = jax.tree_util.tree_flatten(params)
        return {
            "momentum": treedef.unflatten([jnp.zeros_like(p, dtype=jnp.bfloat16) for p in flat]),
            "count": jnp.zeros([], jnp.int32),
        }

    def update_fn(grads, state, params=None):
        flat_grads, treedef = jax.tree_util.tree_flatten(grads)
        flat_bufs, _ = jax.tree_util.tree_flatten(state["momentum"])

        new_flat_updates, new_flat_bufs = [], []
        for g, buf in zip(flat_grads, flat_bufs):
            updated, new_buf = muon_update(g, buf, beta=beta, ns_steps=ns_steps, nesterov=True)
            new_flat_updates.append(updated)
            new_flat_bufs.append(new_buf)

        return (
            treedef.unflatten(new_flat_updates),
            {"momentum": treedef.unflatten(new_flat_bufs), "count": state["count"] + 1},
        )

    return GradientTransformation(init_fn, update_fn)


def make_muon_optimizer(
    lr: float,
    min_lr: float,
    muon_lr: float,
    warmup_steps: int,
    hold_steps: int,
    total_steps: int,
    weight_decay: float = 0.01,
    muon_beta: float = 0.95,
    ns_steps: int = 5,
    adam_b1: float = 0.9,
    adam_b2: float = 0.95,
    adam_eps: float = 1e-8,
    max_grad_norm: float = 1.0,
) -> GradientTransformation:

    lr_schedule = optax.join_schedules(
        schedules=[
            optax.linear_schedule(0.0, lr, warmup_steps),
            optax.constant_schedule(lr),
            optax.cosine_decay_schedule(
                lr,
                total_steps - warmup_steps - hold_steps,
                alpha=min_lr / lr,
            ),
        ],
        boundaries=[warmup_steps, warmup_steps + hold_steps],
    )

    muon_tx = optax.chain(
        optax.clip_by_global_norm(max_grad_norm),
        _muon_transform(beta=muon_beta, ns_steps=ns_steps),
        optax.scale_by_learning_rate(lr_schedule),
    )

    adamw_tx = optax.chain(
        optax.clip_by_global_norm(max_grad_norm),
        optax.scale_by_adam(b1=adam_b1, b2=adam_b2, eps=adam_eps),
        optax.add_decayed_weights(weight_decay),
        optax.scale_by_learning_rate(lr_schedule),
    )

    def param_label(path, param):
        path_str = "/".join(str(p.key) for p in path)
        excluded = ("tok_emb", "lm_head", "embed")
        if param.ndim == 2 and not any(e in path_str for e in excluded):
            return "muon"
        return "adamw"

    return optax.multi_transform(
        {"muon": muon_tx, "adamw": adamw_tx},
        param_labels=lambda params: jax.tree_util.tree_map_with_path(param_label, params),
    )

def update_fn(grads, state, params=None):
    new_updates = {}
    new_bufs = {}

    flat_grads, treedef = jax.tree_util.tree_flatten(grads)
    flat_bufs, _ = jax.tree_util.tree_flatten(state["momentum"])

    new_flat_updates = []
    new_flat_bufs = []
    for g, buf in zip(flat_grads, flat_bufs):
        updated, new_buf = muon_update(g, buf, beta=beta, ns_steps=ns_steps, nesterov=True)
        new_flat_updates.append(updated)
        new_flat_bufs.append(new_buf)

    updates = treedef.unflatten(new_flat_updates)
    new_momentum = treedef.unflatten(new_flat_bufs)

    return updates, {"momentum": new_momentum, "count": state["count"] + 1}



class BaselineTokenizer:
    def __init__(self, tokenizer_name: str = "gpt2", max_length: int = 4097):
        from transformers import AutoTokenizer

        self.tokenizer: PreTrainedTokenizerBase = AutoTokenizer.from_pretrained(
            tokenizer_name,
            use_fast=True,
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.max_length = max_length

    @property
    def vocab_size(self) -> int:
        return len(self.tokenizer)

    def encode_batch(self, texts: list[str]) -> dict[str, jnp.ndarray]:
        encoded = self.tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="np",
        )

        input_ids = jnp.asarray(encoded["input_ids"], dtype=jnp.int32)
        attention_mask = jnp.asarray(encoded["attention_mask"], dtype=jnp.float32)

        return {
            "tokens": input_ids[:, :-1],
            "labels": input_ids[:, 1:],
            "loss_mask": attention_mask[:, 1:],
        }


def build_trainer(**overrides: Any) -> "Trainer":
    config = ModelArg()
    for key, value in overrides.items():
        if not hasattr(config, key):
            raise ValueError(f"Unknown config field: {key}")
        setattr(config, key, value)
    return Trainer(config)


def clean_text(text: str) -> str:
    return " ".join(text.split()).strip()


def load_fineweb_edu(config: ModelArg) -> IterableDataset:
    from datasets import load_dataset

    dataset = load_dataset(
        config.dataset_name,
        name=config.dataset_subset,
        split=config.dataset_split,
        streaming=config.streaming,
    )
    dataset = dataset.map(
        lambda row: {config.text_field: clean_text(row.get(config.text_field, ""))}
    )
    dataset = dataset.filter(lambda row: bool(row[config.text_field]))

    if config.shuffle_buffer_size > 0:
        dataset = dataset.shuffle(
            seed=config.seed,
            buffer_size=config.shuffle_buffer_size,
        )

    if config.max_examples is not None:
        dataset = dataset.take(config.max_examples)

    return dataset


def iter_text_batches(
    dataset: IterableDataset,
    batch_size: int,
    text_field: str = "text",
) -> Iterator[list[str]]:
    batch: list[str] = []
    for row in dataset:
        batch.append(row[text_field])
        if len(batch) == batch_size:
            yield batch
            batch = []

    if batch:
        yield batch


def build_fineweb_edu_text_pipeline(config: ModelArg) -> Iterator[list[str]]:
    dataset = load_fineweb_edu(config)
    return iter_text_batches(
        dataset=dataset,
        batch_size=config.batch_size,
        text_field=config.text_field,
    )


def cross_entropy_loss(
    logits: jnp.ndarray,
    labels: jnp.ndarray,
    loss_mask: jnp.ndarray,
) -> jnp.ndarray:
    losses = optax.softmax_cross_entropy_with_integer_labels(logits, labels)
    loss_mask = loss_mask.astype(jnp.float32)
    return (losses * loss_mask).sum() / jnp.maximum(loss_mask.sum(), 1.0)


class Trainer:
    def __init__(self, args: ModelArg):
        self.args = args

        self.lr_schedule = optax.warmup_cosine_decay_schedule(
            init_value=0.0,
            peak_value=self.args.lr,
            warmup_steps=self.args.warmup_steps,
            decay_steps=max(self.args.total_steps, self.args.warmup_steps + 1),
            end_value=self.args.min_lr,
        )
        self.lr_schedule = optax.join_schedules(
            schedules=[
                optax.linear_schedule(0.0, self.args.lr, self.args.warmup_steps),
                optax.constant_schedule(self.args.lr),
                optax.cosine_decay_schedule(self.args.lr, self.args.total_steps // 2,
                                            alpha=self.args.min_lr / self.args.lr),
            ],
            boundaries=[self.args.warmup_steps, self.args.total_steps // 2],
        )

        self.optimizer = make_muon_optimizer(
          lr=self.args.lr,
          muon_lr=self.args.muon_lr,
          min_lr=self.args.min_lr,
          warmup_steps=self.args.warmup_steps,
          hold_steps=self.args.hold_steps,
          total_steps=self.args.total_steps,
          weight_decay=self.args.weight_decay,
          muon_beta=self.args.muon_beta,
          ns_steps=self.args.muon_ns_steps,
          adam_b1=self.args.adam_b1,
          adam_b2=self.args.adam_b2,
          adam_eps=self.args.adam_eps,
          max_grad_norm=self.args.max_grad_norm,
      )

        self.model = None
        self.tokenizer = None
        self.train_data = None
        self.eval_data = None
        self.state = None
        self.wandb = None
        self.wandb_run = None

    def build_tokenizer(self) -> BaselineTokenizer:
        self.tokenizer = BaselineTokenizer(
            tokenizer_name=self.args.tokenizer_name,
            max_length=self.args.max_seq + 1,
        )
        self.args.vocab_size = self.tokenizer.vocab_size
        return self.tokenizer

    def build_model(self) -> Transformer:
        self.model = Transformer(self.args)
        return self.model

    def build_train_data(self) -> Iterator[list[str]]:
        self.train_data = iter(build_fineweb_edu_text_pipeline(self.args))
        return self.train_data

    def setup_wandb(self) -> None:
        if self.wandb_run is not None:
            return

        import wandb

        if "google.colab" in sys.modules:
            wandb.login(relogin=True)
        else:
            wandb.login(relogin=False)

        self.wandb = wandb
        self.wandb_run = wandb.init(
            project=self.args.wandb_project,
            entity=self.args.wandb_entity,
            name=self.args.wandb_run_name,
            config=asdict(self.args),
        )

    def setup(self) -> dict[str, Any]:
        self.build_tokenizer()
        self.build_model()
        self.build_train_data()
        self.eval_data = None
        self.setup_wandb()

        rng = jax.random.PRNGKey(self.args.seed)
        init_tokens = jnp.zeros((1, self.args.max_seq), dtype=jnp.int32)
        variables = self.model.init(rng, init_tokens)
        params = variables["params"]

        self.state = {
            "params": params,
            "opt_state": self.optimizer.init(params),
            "step": 0,
            "rng": rng,
        }
        return self.state

    def make_batch(self, texts: list[str]) -> dict[str, jnp.ndarray]:
        if self.tokenizer is None:
            self.build_tokenizer()
        return self.tokenizer.encode_batch(texts)

    def log_metrics(self, metrics: dict[str, float]) -> None:
        if self.wandb_run is not None:
            self.wandb.log(metrics, step=self.state["step"])

    def train(self) -> dict[str, Any]:
      if self.state is None:
        self.setup()

      # Define train_step ONCE outside the loop
      @jax.jit
      def train_step(params, opt_state, tokens, labels, loss_mask):
          def loss_fn(params):
              logits = self.model.apply({"params": params}, tokens)
              return cross_entropy_loss(logits, labels, loss_mask)
          loss, grads = jax.value_and_grad(loss_fn)(params)
          updates, opt_state_new = self.optimizer.update(grads, opt_state, params)
          new_params = optax.apply_updates(params, updates)
          return new_params, opt_state_new, loss

      for _ in range(self.args.total_steps):
          try:
              texts = next(self.train_data)
          except StopIteration:
              break

          batch = self.make_batch(texts)

          new_params, new_opt_state, loss = train_step(
              self.state["params"],
              self.state["opt_state"],
              batch["tokens"],
              batch["labels"],
              batch["loss_mask"],
          )

          self.state["params"] = new_params
          self.state["opt_state"] = new_opt_state
          self.state["step"] += 1
          self.state["loss"] = loss
          self.state["learning_rate"] = self.lr_schedule(self.state["step"])

          if self.state["step"] % self.args.log_every_steps == 0:
              self.log_metrics({
                  "train/loss": float(loss),
                  "train/learning_rate": float(self.state["learning_rate"]),
              })

      return self.state

    def evaluate(self, max_batches: int = 10) -> dict[str, float]:
        if self.state is None:
            self.setup()

        eval_iter = iter(build_fineweb_edu_text_pipeline(self.args))
        total_loss = 0.0
        seen = 0

        for _ in range(max_batches):
            try:
                texts = next(eval_iter)
            except StopIteration:
                break

            batch = self.make_batch(texts)
            logits = self.model.apply({"params": self.state["params"]}, batch["tokens"])
            loss = cross_entropy_loss(logits, batch["labels"], batch["loss_mask"])
            total_loss += float(loss)
            seen += 1

        metrics = {
            "eval_loss": total_loss / max(seen, 1),
            "eval_batches": float(seen),
        }
        self.state["eval_loss"] = metrics["eval_loss"]

        if self.wandb_run is not None:
            self.wandb.log(
                {f"eval/{k}": v for k, v in metrics.items()},
                step=self.state["step"],
            )

        return metrics

    def save(self, path: str = "checkpoints/latest.pkl") -> Path:
        if self.state is None:
            raise ValueError("Call setup() or train() before save().")

        checkpoint_path = Path(path)
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "args": asdict(self.args),
            "state": self.state,
        }
        with checkpoint_path.open("wb") as handle:
            pickle.dump(payload, handle)
        return checkpoint_path

    def load(self, path: str = "checkpoints/latest.pkl") -> dict[str, Any]:
        checkpoint_path = Path(path)
        with checkpoint_path.open("rb") as handle:
            payload = pickle.load(handle)

        self.args = ModelArg(**payload["args"])
        self.build_tokenizer()
        self.build_model()
        self.build_train_data()
        self.state = payload["state"]
        return self.state

### Training

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=memory.free,memory.total',
                        '--format=csv,noheader'], capture_output=True, text=True)
print(result.stdout)

jax.clear_caches()
import gc; gc.collect()

trainer = build_trainer(
    batch_size=128,
    max_seq=1024,
    n_layer_groups=4,
    total_steps=15000,
    vocab_size=50257,
    lr=1e-4,
    min_lr=1e-5,
    warmup_steps=300,
    log_every_steps=10,
    dataset_subset="sample-10BT",
    tokenizer_name="gpt2",
    wandb_project="attn-residual",
    wandb_run_name="baseline-debug",
)

trainer.setup()

trainer.train()

trainer.evaluate()

trainer.save("checkpoints/baseline-latest.pkl")